# CocoAnalytica: Official Model Training and Prediction Inspection

This is the official editable workflow for preparing data, training EfficientNetB0, exporting the model used for AI Analysis, and inspecting the prediction score for each image in the held-out `test` folder. F1-score and confusion-matrix reporting lives in `model-f1-matrices.ipynb` so it cannot be confused with the training workflow.

The deployment artifact is TensorFlow Lite (`model_outputs/coconut_leaf_multilabel_cnn.tflite`), which is the artifact copied into Render by the Dockerfile.

## Current dataset revision

Only visually reviewed field views remain in the four source folders. Source counts:
Healthy **120**, Yellowing **56**, CSI **155**, Rhino **362**. This review concerns
framing, not expert confirmation of disease labels. All `legacy_unique` and added-source
subfolders have been flattened after review. Removed photos, old splits, candidate
models and the unused non-palm collection are recoverable in
`../dataset_archive/broad_view_20261008/`.

Previous outputs were cleared because they describe the older dataset. Rerun preparation
and train from ImageNet before reporting metrics for this revision. Saved augmentation
preserves the whole field of view using flips and mild brightness/contrast changes.
The currently deployed model is unchanged until training, release, and deployment.


In [ ]:
from __future__ import annotations
import json
import shutil
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import tensorflow as tf
from IPython.display import display, Markdown

PROJECT_ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                    if (p / 'backend/app/ml/predictor.py').is_file())
sys.path.insert(0, str(PROJECT_ROOT / 'backend'))
sys.path.insert(0, str(PROJECT_ROOT / 'backend/scripts'))
from app.ml import predictor
from notebook_workflow import prepare_dataset, file_hash

MODEL_DIR = PROJECT_ROOT / 'Thesis AI Model'
OUTPUT_DIR = MODEL_DIR / 'model_outputs'
CANDIDATE_ROOT = OUTPUT_DIR / 'candidates'
LIVE_MODEL_PATH = OUTPUT_DIR / 'coconut_leaf_multilabel_cnn.keras'
LIVE_TFLITE_PATH = OUTPUT_DIR / 'coconut_leaf_multilabel_cnn.tflite'
LIVE_LABEL_CONFIG_PATH = OUTPUT_DIR / 'label_config.json'
CLASS_NAMES = ['Healthy', 'Yellowing', 'Coconut_Scale_Insect', 'Rhinoceros_Beetle']
IMAGE_SIZE, BATCH_SIZE, SEED = (224, 224), 8, 20261008
TRAIN_IMAGES_PER_CLASS = 700  # Originals + exported variants; never truncates larger classes.
tf.config.threading.set_intra_op_parallelism_threads(2)
tf.config.threading.set_inter_op_parallelism_threads(2)
tf.keras.utils.set_random_seed(SEED)
print('Official training notebook:', MODEL_DIR / 'model.ipynb')
print('Separate F1/matrix notebook:', MODEL_DIR / 'model-f1-matrices.ipynb')
print('Keras reference model:', LIVE_MODEL_PATH)
print('TensorFlow Lite artifact loaded by Render:', LIVE_TFLITE_PATH)

## Prepare and inspect exported augmentation
Rerunning with the same source and settings reuses a verified dataset. Changing settings creates a new version; originals and old splits are preserved.

In [ ]:
DATASET_DIR, manifest = prepare_dataset(MODEL_DIR, TRAIN_IMAGES_PER_CLASS, SEED)
print('Dataset:', DATASET_DIR)
print(json.dumps(manifest['counts'], indent=2))
print('Source duplicate copies retained:', manifest['duplicate_copies_retained_in_source'])
print('Skipped files:', len(manifest['skipped']))
print('Conflicting-label copies excluded for review:', len(manifest['excluded_conflicting_labels']))
print('Existing split overlaps corrected:', len(manifest['repaired_pixel_duplicate_leakage']))
print(manifest['limitation'])
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, label in zip(axes, CLASS_NAMES):
    samples = sorted((DATASET_DIR / 'train' / label).glob('aug_*.jpg'))
    if not samples:
        samples = sorted((DATASET_DIR / 'train' / label).glob('*'))
    ax.imshow(plt.imread(samples[0])); ax.set_title(label.replace('_', ' ')); ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# Dataset split table: derived from the verified manifest above.
def display_dataset_split_table(counts):
    lines = [
        "### Dataset Split Summary",
        "",
        "| Category | Train | Validation | Test |",
        "| --- | ---: | ---: | ---: |",
    ]
    train_total = validation_total = test_total = 0
    for label in CLASS_NAMES:
        train_row = counts["train"][label]
        validation_row = counts["validation"][label]
        test_row = counts["test"][label]
        train_total += train_row["total"]
        validation_total += validation_row["total"]
        test_total += test_row["total"]
        train_text = f"{train_row['originals']} real + {train_row['augmented']} augmented = {train_row['total']}"
        lines.append(
            f"| {label.replace('_', ' ')} | {train_text} | "
            f"{validation_row['total']} real | {test_row['total']} real |"
        )
    lines.append(f"| **Total** | **{train_total:,}** | **{validation_total:,}** | **{test_total:,}** |")
    display(Markdown("\n".join(lines)))


display_dataset_split_table(manifest["counts"])


In [ ]:
dataset_args = dict(
    labels="inferred",
    label_mode="int",
    class_names=CLASS_NAMES,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
)

train = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR / "train", shuffle=True, seed=SEED, **dataset_args
)
validation = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR / "validation", shuffle=False, **dataset_args
)
test = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR / "test", shuffle=False, **dataset_args
)

options = tf.data.Options()
options.threading.private_threadpool_size = 2
options.threading.max_intra_op_parallelism = 1
# The directory loader already shuffles; keep only one prefetched batch.
train = train.with_options(options).prefetch(1)
validation = validation.with_options(options).prefetch(1)
test = test.with_options(options).prefetch(1)

## CNN Model Architecture

### Purpose

The CocoAnalytica model is a four-class convolutional neural network (CNN) that assists in classifying a coconut leaf image into one of the following conditions: **Healthy**, **Yellowing**, **Coconut Scale Insect (CSI)**, or **Rhinoceros Beetle**. It is an AI-assisted decision-support tool; its prediction should be reviewed together with field observations and officer validation.

### Input Image

Each uploaded coconut leaf photograph is converted to a three-channel RGB image and resized to **224 × 224 pixels**. This provides a consistent image size for all training and website predictions.

### Training Data Augmentation

To reduce overfitting while preserving the broad field view of the farm photographs, augmentation is applied only to the training images. The training images may be horizontally flipped, slightly rotated, and given a small contrast adjustment. No artificial crop or zoom is used. Validation and test images are never augmented, so their results remain based on real photographs that were not used as generated variants during training.

### Feature Extraction Backbone

The model uses **EfficientNetB0** as its CNN backbone. EfficientNetB0 is pretrained using ImageNet, which allows the model to begin with general visual features such as lines, edges, textures, shapes, and colour patterns. The original EfficientNet classification layer is removed and replaced with a custom CoconutAnalytica classification head.

During the first training stage, the EfficientNetB0 backbone is frozen while the custom classification head learns the four coconut-leaf conditions. During fine-tuning, the final 40 EfficientNetB0 layers are made trainable so the model can adapt its higher-level features to coconut leaf conditions. Batch Normalization layers remain frozen during fine-tuning for stable training.

### Classification Head

The EfficientNetB0 feature maps are passed through the following layers:

1. **Global Average Pooling** converts the extracted feature maps into a compact feature vector.
2. **Batch Normalization** stabilizes the values passed to the classifier.
3. **Dense Layer with 256 neurons and ReLU activation** learns class-specific patterns from the extracted leaf features.
4. **Dropout at 30%** randomly disables some training connections to reduce overfitting.
5. **Output Layer with 4 neurons and Softmax activation** produces one probability for each condition. The four probabilities sum to 100%, and the highest probability is the predicted condition.

### Training and Prediction

The model is trained with the Adam optimizer and sparse categorical cross-entropy loss. The classification head is trained first with a learning rate of 0.001, then the final EfficientNetB0 layers are fine-tuned with a learning rate of 0.00001. Early stopping and learning-rate reduction are used to avoid unnecessary training after validation performance stops improving.

For website prediction, the model analyses both the original uploaded image and a horizontally flipped version, then averages their probabilities. Validation-derived temperature calibration, a minimum confidence threshold, and a top-two probability margin are used to flag uncertain predictions. The final output remains one of the four confirmed coconut conditions; non-palm detection is intentionally not included until a sufficiently reviewed non-palm dataset is available.

### Architecture Flow

**224 × 224 RGB coconut leaf image** → **training augmentation** → **EfficientNetB0 pretrained feature extractor** → **Global Average Pooling** → **Batch Normalization** → **Dense 256 (ReLU)** → **Dropout 30%** → **Dense 4 (Softmax)** → **Healthy / Yellowing / CSI / Rhinoceros Beetle**


## Editable model and training settings
EfficientNetB0 uses ImageNet transfer learning and a four-class softmax head. Exported augmentation is supplemented by mild random training augmentation. Keep validation/test unaugmented. Select experiments using validation, then report the final test results once.

In [ ]:
EXPERIMENT = {
    "name": "head_256_finetune_40",
    "head_units": 256,
    "dropout": 0.30,
    "fine_tune_layers": 40,
    "head_epochs": 18,
    "fine_tune_epochs": 12,
    "head_learning_rate": 1e-3,
    "fine_tune_learning_rate": 1e-5,
    "loss": "cross_entropy",  # Change to "focal" only for a separate comparison run.
}

augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip("horizontal", seed=SEED),
        tf.keras.layers.RandomRotation(0.06, seed=SEED),
        # Preserve the broad field of view: no random zoom crop.
        tf.keras.layers.RandomContrast(0.10, seed=SEED),
    ],
    name="field_photo_augmentation",
)

class SparseCategoricalFocalLoss(tf.keras.losses.Loss):
    def __init__(self, gamma: float = 1.5, name: str = "sparse_categorical_focal_loss"):
        super().__init__(name=name)
        self.gamma = gamma

    def call(self, y_true, y_pred):
        y_true = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
        y_pred = tf.clip_by_value(y_pred, tf.keras.backend.epsilon(), 1.0)
        selected = tf.gather(y_pred, y_true, axis=1, batch_dims=1)
        return -tf.pow(1.0 - selected, self.gamma) * tf.math.log(selected)


def build_model(config: dict[str, object]) -> tuple[tf.keras.Model, tf.keras.Model]:
    base = tf.keras.applications.EfficientNetB0(
        include_top=False,
        weights="imagenet",
        input_shape=(*IMAGE_SIZE, 3),
    )
    base.trainable = False

    inputs = tf.keras.Input(shape=(*IMAGE_SIZE, 3), name="image")
    x = augmentation(inputs)
    x = tf.keras.applications.efficientnet.preprocess_input(x)
    x = base(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D(name="global_average_pooling")(x)
    x = tf.keras.layers.BatchNormalization(name="head_batch_normalization")(x)
    x = tf.keras.layers.Dense(int(config["head_units"]), activation="relu", name="classification_features")(x)
    x = tf.keras.layers.Dropout(float(config["dropout"]), name="head_dropout")(x)
    outputs = tf.keras.layers.Dense(len(CLASS_NAMES), activation="softmax", name="condition")(x)
    return tf.keras.Model(inputs, outputs, name="cocoanalytics_efficientnetb0"), base


def choose_loss(name: str):
    if name == "focal":
        return SparseCategoricalFocalLoss()
    return tf.keras.losses.SparseCategoricalCrossentropy()


def compile_model(model: tf.keras.Model, learning_rate: float, loss_name: str) -> None:
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss=choose_loss(loss_name),
        metrics=["accuracy"],
    )

In [ ]:
def prediction_rows(model_path, config_path, dataset_root, split='test', runtime='tflite'):
    """Run each held-out image through the same predictor used by the website."""
    predictor._model = None
    predictor._config = None
    predictor._load_error = None
    predictor._loaded_identity = None
    predictor._loaded_runtime = None
    rows = []
    source_root = Path(dataset_root) / split
    for actual_label in CLASS_NAMES:
        for image_path in sorted((source_root / actual_label).glob('*')):
            if not image_path.is_file():
                continue
            result = predictor.predict_upload_file(
                image_path.read_bytes(),
                model_path=Path(model_path),
                config_path=Path(config_path),
                runtime=runtime,
                max_bytes=max(image_path.stat().st_size, 10 * 1024 * 1024),
            )
            rows.append({
                'file': str(image_path),
                'actual': actual_label,
                'predicted': result.label.replace(' ', '_'),
                'confidence': result.confidence,
                'uncertain': result.uncertain,
                'scores': result.predictions,
            })
    return rows


def display_prediction_scores(rows, title):
    """Show each image's actual label, prediction, confidence, and four class scores."""
    lines = [
        f'### Per-Image Prediction Scores - {title}',
        '',
        '| Image | Actual | Predicted | Confidence | Healthy | Yellowing | CSI | Rhinoceros Beetle |',
        '| --- | --- | --- | ---: | ---: | ---: | ---: | ---: |',
    ]
    for row in rows:
        scores = row['scores']
        score = lambda label: scores.get(label, scores.get(label.replace('_', ' '), 0.0))
        lines.append(
            f"| {Path(row['file']).name} | {row['actual'].replace('_', ' ')} | "
            f"{row['predicted'].replace('_', ' ')} | {row['confidence']:.1f}% | "
            f"{score('Healthy'):.1f}% | {score('Yellowing'):.1f}% | "
            f"{score('Coconut_Scale_Insect'):.1f}% | {score('Rhinoceros_Beetle'):.1f}% |"
        )
    display(Markdown('\n'.join(lines)))

In [ ]:
run_id = f"{EXPERIMENT['name']}_{datetime.now():%Y%m%d-%H%M%S}"
run_dir = CANDIDATE_ROOT / run_id
run_dir.mkdir(parents=True, exist_ok=True)

model, base_model = build_model(EXPERIMENT)
compile_model(model, float(EXPERIMENT["head_learning_rate"]), str(EXPERIMENT["loss"]))

best_checkpoint = tf.keras.callbacks.ModelCheckpoint(
    run_dir / "best.weights.h5", monitor="val_loss", save_best_only=True, save_weights_only=True
)
callbacks = [
    best_checkpoint,
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-7),
    tf.keras.callbacks.CSVLogger(run_dir / "head_training_log.csv"),
]

head_history = model.fit(
    train,
    validation_data=validation,
    epochs=int(EXPERIMENT["head_epochs"]),
    callbacks=callbacks,
    verbose=2,
)

base_model.trainable = True
fine_tune_at = max(len(base_model.layers) - int(EXPERIMENT["fine_tune_layers"]), 0)
for index, layer in enumerate(base_model.layers):
    layer.trainable = index >= fine_tune_at
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

compile_model(model, float(EXPERIMENT["fine_tune_learning_rate"]), str(EXPERIMENT["loss"]))
fine_tune_history = model.fit(
    train,
    validation_data=validation,
    epochs=len(head_history.history["loss"]) + int(EXPERIMENT["fine_tune_epochs"]),
    initial_epoch=len(head_history.history["loss"]),
    verbose=2,
    callbacks=[
        best_checkpoint,
            tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-7),
        tf.keras.callbacks.CSVLogger(run_dir / "fine_tune_log.csv"),
    ],
)
# Retain the best validation-loss weights across both training phases.
model.load_weights(run_dir / "best.weights.h5")
print("Best validation loss across both phases:", float(best_checkpoint.best))


## Calibrate, export, and inspect a candidate

Temperature is fitted on validation with the same flipped-image averaging used by the API. It changes confidence, not the winning class. The Keras candidate is exported to TensorFlow Lite, then each image in the held-out `test` folder is shown with its website-runtime prediction scores. F1 scores and confusion matrices are intentionally kept in `model-f1-matrices.ipynb`.

In [ ]:
def collect_predictions(model, dataset):
    probabilities, labels = [], []
    for images, batch_labels in dataset:
        probabilities.append(model(images, training=False).numpy())
        labels.append(batch_labels.numpy())
    return np.concatenate(probabilities), np.concatenate(labels)


def test_time_augmented_predictions(model, dataset):
    probabilities, labels = [], []
    for images, batch_labels in dataset:
        original = model(images, training=False)
        flipped = model(tf.image.flip_left_right(images), training=False)
        probabilities.append(((original + flipped) / 2.0).numpy())
        labels.append(batch_labels.numpy())
    return np.concatenate(probabilities), np.concatenate(labels)


def accuracy_from_probabilities(probabilities, labels):
    return float(np.mean(np.argmax(probabilities, axis=1) == labels))


def fit_temperature(probabilities, labels):
    """Fit temperature on validation only; lower validation cross-entropy wins."""
    temperatures = np.linspace(0.5, 3.0, 51)
    logits = np.log(np.clip(probabilities, 1e-7, 1.0))
    losses = []
    for temperature in temperatures:
        scaled = logits / temperature
        scaled -= scaled.max(axis=1, keepdims=True)
        calibrated = np.exp(scaled)
        calibrated /= calibrated.sum(axis=1, keepdims=True)
        losses.append(-np.mean(np.log(np.clip(calibrated[np.arange(len(labels)), labels], 1e-7, 1.0))))
    return float(temperatures[int(np.argmin(losses))])


validation_probabilities, validation_labels = test_time_augmented_predictions(model, validation)
calibration_temperature = fit_temperature(validation_probabilities, validation_labels)
candidate_model_path = run_dir / 'coconut_leaf_model.keras'
model.save(candidate_model_path)
label_config = {
    'class_names': CLASS_NAMES, 'thresholds': {name: .5 for name in CLASS_NAMES},
    'uncertain_threshold': .5, 'min_top_two_margin': .10,
    'temperature': calibration_temperature, 'image_size': list(IMAGE_SIZE),
    'run_id': run_id, 'source_notebook': 'Thesis AI Model/model.ipynb',
}
candidate_config = run_dir / 'label_config.json'
candidate_config.write_text(json.dumps(label_config, indent=2), encoding='utf-8')
candidate_tflite_path = run_dir / 'coconut_leaf_model.tflite'
converter = tf.lite.TFLiteConverter.from_keras_model(tf.keras.models.load_model(candidate_model_path, compile=False))
candidate_tflite_path.write_bytes(converter.convert())
print('Saved Keras reference:', candidate_model_path)
print('Saved TensorFlow Lite deployment artifact:', candidate_tflite_path)

In [ ]:
candidate_identity = {
    'model_sha256': file_hash(candidate_tflite_path),
    'config_sha256': file_hash(candidate_config),
    'run_id': run_id,
    'source_notebook': 'Thesis AI Model/model.ipynb',
    'runtime': 'tflite',
}
candidate_test_predictions = prediction_rows(
    candidate_tflite_path, candidate_config, DATASET_DIR, split='test', runtime='tflite'
)
(run_dir / 'test_predictions_per_image.json').write_text(
    json.dumps({**candidate_identity, 'predictions': candidate_test_predictions}, indent=2),
    encoding='utf-8',
)
display_prediction_scores(candidate_test_predictions, 'Candidate TensorFlow Lite model, held-out test folder')
print('Saved per-image test predictions:', run_dir / 'test_predictions_per_image.json')
print('Model fingerprint:', candidate_identity['model_sha256'])

## Release the reviewed model for AI Analysis

Set `RELEASE_TO_WEBSITE = True` after reviewing the per-image held-out-test predictions. Release is an explicit decision and does not alter the model artifact. Use `model-f1-matrices.ipynb` for all F1-score and confusion-matrix reporting.

In [ ]:
RELEASE_TO_WEBSITE = False  # Change only after reviewing the per-image test predictions above.
if RELEASE_TO_WEBSITE:
    if (file_hash(candidate_tflite_path) != candidate_identity['model_sha256'] or
            file_hash(candidate_config) != candidate_identity['config_sha256']):
        raise RuntimeError('Candidate changed after test predictions were generated. Rerun the inspection cell.')
    backup = CANDIDATE_ROOT / ('before_release_' + datetime.now().strftime('%Y%m%d-%H%M%S-%f'))
    backup.mkdir(parents=True)
    for name in ['coconut_leaf_multilabel_cnn.keras', 'coconut_leaf_multilabel_cnn.tflite', 'label_config.json']:
        current = OUTPUT_DIR / name
        if current.exists():
            shutil.copy2(current, backup / name)
    shutil.copy2(candidate_model_path, LIVE_MODEL_PATH)
    shutil.copy2(candidate_tflite_path, LIVE_TFLITE_PATH)
    shutil.copy2(candidate_config, LIVE_LABEL_CONFIG_PATH)
    shutil.copy2(run_dir / 'test_predictions_per_image.json', OUTPUT_DIR / 'test_predictions_per_image.json')
    print('Released locally:', candidate_identity['model_sha256'])
    print('Commit/push and wait for Render to redeploy, then compare /predict/status model_sha256.')
else:
    print('Candidate saved. Review per-image test predictions before enabling release.')

## Inspect the currently installed Render website model without retraining

Run setup and dataset preparation, then this cell to inspect the active local TensorFlow Lite artifact on every held-out `test` image. It uses the deployed website prediction path and reports each image's predicted condition and four class scores.

In [ ]:
active_test_predictions = prediction_rows(
    LIVE_TFLITE_PATH, LIVE_LABEL_CONFIG_PATH, DATASET_DIR, split='test', runtime='tflite'
)
display_prediction_scores(active_test_predictions, 'Currently installed Render TensorFlow Lite model, held-out test folder')
print('Model fingerprint:', file_hash(LIVE_TFLITE_PATH))
print('The Render /predict/status fingerprint must match before calling this the deployed result.')

## F1 score and confusion-matrix reporting

F1 scores and confusion matrices are kept in the separate `model-f1-matrices.ipynb` notebook. It presents a balanced 20-images-per-condition evaluation, combining all approved external photos with original held-out internal test images to fill each class to 20. Keeping it separate prevents training and reporting code from being mixed.

The evaluation notebook never duplicates an image to balance a class. It records how many approved external and held-out internal test images were used for each condition; `Y04` is excluded because it is a pixel-identical duplicate.

## Connected files: from training to the website

**Training/export:** `model.ipynb` -> Keras reference -> `model_outputs/coconut_leaf_multilabel_cnn.tflite` plus `label_config.json`.

**Evaluation:** `model-f1-matrices.ipynb` -> approved external images plus held-out internal test images -> balanced 20-per-class F1 table and confusion matrix.

**Upload/request:** Farmer page -> frontend prediction service -> configured API URL -> FastAPI prediction route -> predictor -> exported TensorFlow Lite model -> JSON scores -> UI.

| File (click to open) | Responsibility |
|---|---|
| [model.ipynb](model.ipynb) | Official training, per-image test-folder prediction inspection, and release controls. |
| [model-f1-matrices.ipynb](model-f1-matrices.ipynb) | External-only F1-score and confusion-matrix evaluation. |
| [notebook_workflow.py](../backend/scripts/notebook_workflow.py) | Saved augmentation, split provenance, and evaluation using actual API prediction code. |
| [TensorFlow Lite deployment model](model_outputs/coconut_leaf_multilabel_cnn.tflite) | Exact model artifact loaded by the Render backend. |
| [label_config.json](model_outputs/label_config.json) | Class order, image size, temperature, uncertainty thresholds, release run ID. |
| [predictor.py](../backend/app/ml/predictor.py) | **Direct connection to the AI model:** loads the runtime selected by `ML_RUNTIME`, decodes/resizes photos, averages flipped predictions, and calibrates confidence. |
| [predict.py](../backend/app/api/predict.py) | **API connection:** `/predict`, `/predict/batch`, `/predict/status`; routes uploads to predictor. |
| [config.py](../backend/app/core/config.py) | Resolves `ML_MODEL_PATH` and `ML_LABEL_CONFIG_PATH`. |
| [main.py](../backend/app/main.py) | Starts FastAPI and registers API routes. |
| [frontend predict.ts](../frontend/src/services/predict.ts) | Sends photos to API, maps responses, computes batch majority and average winning-class confidence; may compress large uploads. |
| [frontend api.ts](../frontend/src/services/api.ts) | Uses `VITE_API_URL` to connect the Vercel frontend to the Render backend with authentication. |
| [Dockerfile](../Dockerfile) | Copies the TensorFlow Lite model and label config into `/app/model_outputs` and launches FastAPI. |
| [render.yaml](../render.yaml) | Render Free service configuration and TensorFlow Lite runtime settings. |

Render model path: `/app/model_outputs/coconut_leaf_multilabel_cnn.tflite`.
Local TensorFlow Lite artifact: `Thesis AI Model/model_outputs/coconut_leaf_multilabel_cnn.tflite`.
The old `backend/scripts/train_balanced_cnn.py` is a legacy experiment tool; this notebook is the official workflow. Neither training tool executes during uploads.

For a defense demonstration: inspect a per-image score in this notebook, open Render's `/predict/status` to compare its model hash, then use the separate evaluation notebook for the balanced 20-per-class F1 scores and confusion matrix. Confidence on a newly uploaded photo is not test accuracy or F1. Website image compression and genuinely different photos can also change individual predictions.